# Mesh generation - Triangle & Voronoi

Build **triangular** and **Voronoi** unstructured (DISV) grids over the watershed with the Triangle program: seed nodes along the river and over the area of interest, triangulate inside the boundary, then derive the Voronoi grid from the same triangulation.

Part of the **mesh-generation** series, adapted from the FloPy watershed geoprocessing example (Hughes and others, 2023, *FloPy Workflows for Creating Structured and Unstructured MODFLOW Models*, Groundwater, https://doi.org/10.1111/gwat.13327). Each notebook builds one family of grids over the same synthetic watershed, samples a fine topographic raster onto the grid, and intersects the river network with the grid cells.

- **Rectilinear (DIS + LGR)** (`mesh-generation-rectilinear`) - structured grids: constant and variable spacing, plus local grid refinement
- **Gridgen quadtree** (`mesh-generation-gridgen`) - a quadtree unstructured (DISV) grid built with Gridgen
- **Triangle & Voronoi** (`mesh-generation-triangle-voronoi`) - unstructured (DISV) grids built with Triangle and Voronoi *(this notebook)*

## Imports and setup

Import FloPy and the shared watershed setup from `mesh_helpers`: the problem extent and contour levels, the fine topographic raster, the boundary/river geometry, and the `resample_topo` / `river_intersection` / `draw_boundary_river` helpers that every mesh-generation notebook uses.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import flopy.plot.styles as styles
import matplotlib.pyplot as plt
import numpy as np
from mesh_helpers import (
    bp,
    draw_boundary_river,
    levels,
    resample_topo,
    river_intersection,
    sgs,
    vmax,
    vmin,
)
from notebook_helpers import densify_geometry

## Triangular grid

Seed nodes by densifying the river segments and adding a regular set of nodes over the area of interest, then triangulate inside the watershed boundary. Triangle writes to a workspace under `models/`.

In [ ]:
from flopy.utils.triangle import Triangle

triangle_ws = pl.Path("models/mesh-generation-triangle")

# seed nodes: densified river segments + a regular grid over the area of interest
nodes = []
for sg in sgs:
    nodes.extend(densify_geometry(sg, 2000))
d = 5000.0 / 3.0
for x in np.arange(65000.0, 90000.0 + d, d):
    for y in np.arange(40000.0, 60000.0 + d, d):
        nodes.append((x, y))
nodes = np.array(nodes)

triangle_ws.mkdir(parents=True, exist_ok=True)
tri = Triangle(maximum_area=5000 * 5000, angle=30, nodes=nodes, model_ws=triangle_ws)
tri.add_polygon(bp)
tri.build(verbose=False)

triangular_grid = flopy.discretization.VertexGrid(
    vertices=tri.get_vertices(),
    cell2d=tri.get_cell2d(),
    nlay=1,
    ncpl=tri.ncpl,
    idomain=np.ones((1, tri.ncpl), dtype=int),
    top=np.ones(tri.ncpl) * 1000.0,
    botm=np.ones((1, tri.ncpl)) * -100.0,
)

top_tg = resample_topo(triangular_grid)
intersection_tg = river_intersection(triangular_grid, all_intersections=True)

In [ ]:
with styles.USGSMap():
    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax.set_aspect("equal")
    pmv = flopy.plot.PlotMapView(modelgrid=triangular_grid, ax=ax)
    pmv.plot_array(top_tg, ec="0.75", vmin=vmin, vmax=vmax)
    pmv.plot_array(intersection_tg, masked_values=[0], alpha=0.2, cmap="Reds_r")
    pmv.contour_array(top_tg, levels=levels, linewidths=0.3, colors="white")
    pmv.plot_inactive(zorder=100)
    draw_boundary_river(ax)
    ax.set_title("Triangular grid")

## Voronoi grid

The Voronoi grid is the dual of the triangulation - build it directly from the `Triangle` object with `VoronoiGrid`, so the two grids share the same node set.

In [ ]:
from flopy.utils.voronoi import VoronoiGrid

voronoi_obj = VoronoiGrid(tri)
voronoi_grid = flopy.discretization.VertexGrid(
    **voronoi_obj.get_gridprops_vertexgrid(),
    nlay=1,
    idomain=np.ones((1, voronoi_obj.ncpl), dtype=int),
)

top_vg = resample_topo(voronoi_grid)
intersection_vg = river_intersection(voronoi_grid, all_intersections=True)

In [ ]:
with styles.USGSMap():
    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax.set_aspect("equal")
    pmv = flopy.plot.PlotMapView(modelgrid=voronoi_grid, ax=ax)
    pmv.plot_array(top_vg, vmin=vmin, vmax=vmax)
    pmv.plot_array(intersection_vg, masked_values=[0], alpha=0.2, cmap="Reds_r")
    pmv.contour_array(top_vg, levels=levels, linewidths=0.3, colors="0.75")
    pmv.plot_inactive()
    draw_boundary_river(ax)
    ax.set_title("Voronoi grid")

**Recap.** Triangle fills the watershed with triangular cells honoring the river and the area of interest, and `VoronoiGrid` derives the Voronoi dual from the same triangulation - two more unstructured DISV grids over the same domain used by the rectilinear and Gridgen notebooks.